In [1]:
import numpy as np
from tqdm import tqdm
import scipy.linalg as la
import numpy.linalg as npla

import torch
import normflows as nf

import pid
import pid.utils.distributions as dist
from pid.utils.generate import sample_mult_poisson

In [2]:
def get_params():
    lamda_m = np.array([2., 2.])
    lamda_x = np.array([1.,])      # Noise to be added to X: shape (d_X,)
    lamda_y = np.array([1.,])

    w_x1_vals = np.arange(11) / 10
    w_x2 = 0.5
    w_y1 = 0.5
    w_y2 = 0.5

    return lamda_m, lamda_x, lamda_y, w_x1_vals, w_x2, w_y1, w_y2


def pid_flows(base, n_flows, latent_dim, flow_type='Spline'):
    flows = []
    for i in range(n_flows):
        if flow_type == 'Spline':
            flows += [nf.flows.AutoregressiveRationalQuadraticSpline(latent_dim, 2, 128)]
            flows += [nf.flows.LULinearPermute(latent_dim)]
        elif flow_type == 'Planar':
            flows += [nf.flows.Planar((latent_dim,))]
        elif flow_type == 'Radial':
            flows += [nf.flows.Radial((latent_dim,))]
        else:
            raise NotImplementedError

    # Construct flow model
    model = nf.NormalizingFlow(base, flows)
    return model

def objective(sig, hx, hy, dm, dx, dy, reg):
    S = (1 + reg) * np.eye(dx) - sig @ sig.T
    B = hx - sig @ hy
    obj = 0.5 / np.log(2) * npla.slogdet(
        np.eye(dm) + hy.T @ hy + B.T @ la.solve(S, B)
    )[1]
    return obj

In [3]:
num_samples=1000
flow_iter=4000
n_flows=16

enable_cuda=True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
lamda_m, lamda_x, lamda_y, w_x1_vals, w_x2, w_y1, w_y2 = get_params()
dm = lamda_m.size  # Number of dimensions of M
dx = lamda_x.size  # Number of dimensions of X
dy = lamda_y.size  # Number of dimensions of Y

In [6]:
######### Prepare the pid flows #########
base_x = dist.GaussianPCA(dim=dx, latent_dim=dm, sigma=1.)
base_y = dist.GaussianPCA(dim=dy, latent_dim=dm, sigma=1.)
base_m = dist.DiagGaussian(dm, trainable=False)

flow_x = pid_flows(base_x, n_flows, dx).to(device)
flow_y = pid_flows(base_y, n_flows, dy).to(device)
flow_m = pid_flows(base_m, n_flows, dm).to(device)

In [7]:
w_x1 = w_x1_vals[0]
w_x = np.array([[w_x1, w_x2]])  # Binomial thinning weights: shape (dx, dm)
w_y = np.array([[w_y1, w_y2]])

m, x, y = sample_mult_poisson(num_samples, lamda_m, w_x, w_y, lamda_x, lamda_y)  # Shape: (d, n)

######### Train pid flows #########
optimizer_x = torch.optim.Adam(flow_x.parameters(), lr=5e-4, weight_decay=1e-5)
optimizer_y = torch.optim.Adam(flow_y.parameters(), lr=5e-4, weight_decay=1e-5)
optimizer_m = torch.optim.Adam(flow_m.parameters(), lr=5e-4, weight_decay=1e-5)

for it in tqdm(range(flow_iter)):
    flow_x.train(), flow_y.train(), flow_m.train()
    optimizer_x.zero_grad(), optimizer_y.zero_grad(), optimizer_m.zero_grad()

    x_ = torch.from_numpy(x.T).float().to(device)
    y_ = torch.from_numpy(y.T).float().to(device)
    m_ = torch.from_numpy(m.T).float().to(device)

    loss_m = flow_m.forward_kld(m_)
    loss_x = flow_x.forward_kld(x_)
    loss_y = flow_y.forward_kld(y_)

    # Do backprop and optimizer step
    if ~(torch.isnan(loss_x) | torch.isinf(loss_x)):
        loss_x.backward()
        optimizer_x.step()
    if ~(torch.isnan(loss_y) | torch.isinf(loss_y)):
        loss_y.backward()
        optimizer_y.step()
    if ~(torch.isnan(loss_m) | torch.isinf(loss_m)):
        loss_m.backward()
        optimizer_m.step()

100%|██████████| 4000/4000 [19:38<00:00,  3.39it/s]


In [18]:
######### Get params #########
hx = flow_x.q0.get_W(detach=True).cpu().numpy()
hy = flow_y.q0.get_W(detach=True).cpu().numpy()
print(hx, hy)

sig, obj, _ = pid.optimizers.thinpid_exact_pid_minimizer(hx, hy, ret_obj=True, reg=1e-7)
union_info = objective(sig, hx, hy, dm, dx, dy, reg=1e-7)

print(union_info)

[[-0.61796564  0.7226706 ]] [[9.1633415e-01 1.6062602e-05]]
0.798207194086846


In [21]:
from pid.optimizers import exact_gauss_tilde_pid, exact_tilde_union_info_minimizer
from pid.utils import whiten

mxy = np.vstack((m, x, y))
cov = np.corrcoef(mxy) # Shape: (dm+dx+dy, dm+dx+dy)

ret = whiten(cov, dm, dx, dy, ret_channel_params=True)
sig_mxy, hx, hy, hxy, sigxy = ret
print(hx, hy)

sig, obj, _ = exact_tilde_union_info_minimizer(hx, hy, ret_obj=True, reg=1e-7)
union_info = objective(sig, hx, hy, dm, dx, dy, reg=1e-7)

print(union_info)

[[0.04501115 0.57223389]] [[0.51355512 0.54280739]]
0.4341633257361627
